# 附录A：必要数学基础

In [1]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / 'lsq').is_dir():
    if root == root.parent:
        raise RuntimeError('Run inside the repository')
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from lsq import (finite_difference, whiten, gauss_newton,
                 levenberg_marquardt, dogleg, recursive_update)
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({'figure.figsize': (6.4, 3.3), 'font.size': 10,
                     'axes.grid': True, 'savefig.dpi': 130})

这一页是工具箱，不是入场考试。正文遇到陌生符号，再回来找对应工具。第一次先读每节第一段，理解它在记录什么，再看公式。

## 一排数、一张表：向量与矩阵

**向量**就是按顺序放的一排数，例如 `[起点, 速度]`。写作 $x\in\mathbb R^n$ 只是说“这排有 $n$ 个实数”。**维度**就是数量或形状，不必想到另外一个世界。

**矩阵**是一张数表。$A\in\mathbb R^{m\times n}$ 是 $m$ 行 $n$ 列：本书里每行一条观测，每列一个参数的作用。$Ax$ 把参数转成各次预测；$A^T$ 是交换行列。矩阵相乘要求前一个的列数等于后一个的行数，像插头插座需要对得上。Python 的一维数组没有明确行列方向，查看 `.shape` 能帮助防止误接。

## 点积和范数：比较方向，汇总大小

**内积（点积）**把对应项相乘再加起来：$x^Ty=\sum_jx_jy_j$。例如 $(1,2)$ 与 $(3,4)$ 得到 $1\times3+2\times4=11$。点积为零叫正交；对非零向量可理解成垂直。零向量也与所有向量点积为零，但不能给它定义普通方向夹角。

**范数**是给一排数算整体大小的规则。$\|x\|_2=\sqrt{\sum_jx_j^2}$ 就是勾股定理的长度，例如 $(3,4)$ 的长度是 5；$\|x\|_\infty=\max_j|x_j|$ 则取最大绝对值。后者可用来检查每个参数是否都足够小，而不是被其他坐标遮住。

## 导数、梯度、雅可比、Hessian：四种变化记录

**导数**记录一个量稍变时，另一个量变多快；**偏导**先固定其他量，只动其中一个。**梯度**收集“每个参数怎样改变一个总分”；**雅可比**收集“每个参数怎样改变每个残差”；**Hessian** 再记录总分的变化率怎样改变，也就是弯曲情况。

对 $F=\tfrac12r^Tr$，把逐层变化连接起来的链式法则给出 $\nabla F=J^Tr$。足够光滑时，附近目标可以写成

$$F(x+h)=F(x)+g^Th+\frac12h^THh+O(\|h\|^3).$$

右边依次是原分数、斜率作用、弯曲作用和剩下的小误差。$O(\|h\|^3)$ 是误差阶的简写，需要相应光滑条件；仅二阶连续可微时，更一般地写 $o(\|h\|^2)$，表示它相对步长平方越来越不显眼。

## 正定、特征值、奇异值：碗是否朝上，哪些方向看得清

实对称矩阵 $B$ **正定**，指所有非零 $x$ 都满足 $x^TBx>0$，对应二次形状朝上的碗；**半正定**允许某些方向完全平坦。前者二次目标有唯一最低点，后者未必唯一。$J^TJ$ 总半正定，只有 $J$ 满列秩时才正定。

**特征值**描述矩阵在某些特殊方向上的缩放；实对称矩阵特征值全正等价于正定。**奇异值**适用于更一般的矩阵，非负，描述不同参数组合传到观测后有多明显；它们的平方是 $A^TA$ 的特征值。

最大与最小非零奇异值之比描述看得见的那些方向之间的敏感性差别。如果还存在完全看不见的方向，整体逆问题通常称条件数无穷大；不能因为只算了非零那部分的有限比值，就说所有参数都能确定。

In [2]:
A=np.array([[1.,0.],[1.,1.],[1.,2.]])
singular=np.linalg.svd(A,compute_uv=False)
eigen=np.linalg.eigvalsh(A.T@A)
print('singular values squared:',singular**2)
print('eigenvalues (ascending):',eigen)
assert np.allclose(np.sort(singular**2),eigen)

singular values squared: [7.16228 0.83772]
eigenvalues (ascending): [0.83772 7.16228]


## 平均位置与共同波动：期望、方差、协方差

**期望**是在同样规则下反复试验的长期平均位置，不承诺下一次就落在这个数上。**方差**是离平均位置的偏差平方的平均；**标准差**是方差开平方，回到原单位。

**协方差**记录两个量是否倾向一起增减。一张协方差矩阵的对角线放各自方差，其他格子放成对的共同变化。对随机向量 $x$，定义为

$$C=\mathbb E[(x-\mathbb Ex)(x-\mathbb Ex)^T].$$

$\mathbb E$ 表示取期望。如果 $z=Bx$ 是线性换算，波动也一起换算：$\operatorname{Cov}(z)=BCB^T$。比如把米换成毫米，数值放大一千倍，方差会放大一百万倍。

协方差为零叫**不相关**；**独立**则更强，意味着知道一个的结果不能增加对另一个的了解。一般不能把两者混同；联合高斯等附加条件下，零协方差才可推出独立。

## 拿正文里的式子检查一次

问题一：$J$ 是 $20\times3$，$r$ 是 20 个数，$J^Tr$ 有几个数？

提示：先把 $J$ 的行列交换，再检查乘法能否对接。

解答：3 个，对应三个参数的梯度。

问题二：信息不足时，加 $\mu I$ 是否能让局部步好算？是否同时测准了所有参数？

解答：$\mu>0$ 时，$h^T(J^TJ+\mu I)h=\|Jh\|^2+\mu\|h\|^2>0$，局部表变正定，有助于计算。但这是给步子加限制，不是新增观测，原来分不清的参数仍可能分不清。